# Component-Aware Self-Speculative Decoding in Hybrid Language Models## Experimental Notebook[![Paper](https://img.shields.io/badge/arXiv-2026.xxxxx-b31b1b.svg)](https://arxiv.org/)[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)**Research Question:** Can the SSM/linear-attention branch within a hybrid model serve as a zero-cost internal draft model for self-speculative decoding?**Models:**- **Qwen3.5-0.8B** (sequential hybrid: 18 linear + 6 attention layers)- **Falcon-H1-0.5B** (parallel hybrid: 36 SSM + 36 attention per layer)- **Qwen2.5-0.5B** (pure Transformer control)- **Falcon-H1-3B** (parallel hybrid, scale invariance test)**Requirements:** NVIDIA GPU with ≥16GB VRAM, CUDA 11.8+, Python 3.10+**Quick start:**```bashpip install -r requirements.txt# Then run cells sequentially```

In [ ]:
# --- Cell 0A: Setup / package installation ---
# Run once per fresh runtime. Restart afterwards.
import subprocess, sys, os

def install(pkg, extra_args=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
    if extra_args: cmd.extend(extra_args)
    cmd.append(pkg)
    subprocess.check_call(cmd)

install("transformers>=4.45")
install("accelerate")
install("datasets")
install("sentencepiece")
install("protobuf")
install("scipy")

# mamba-ssm and causal-conv1d require CUDA toolkit for compilation
# On RunPod: usually pre-installed. If not, install CUDA dev toolkit first.
try:
    install("causal-conv1d", ["--no-build-isolation"])
    install("mamba-ssm", ["--no-build-isolation"])
    print("mamba-ssm + causal-conv1d installed ✓")
except Exception as e:
    print(f"mamba-ssm not available ({e})")
    print("  → Falcon-H1 will use seq length cap (1024)")
    print("  → To fix: ensure CUDA toolkit is installed (nvcc --version)")

try:
    install("flash-attn", ["--no-build-isolation"])
    print("flash-attn installed ✓")
except Exception:
    print("flash-attn not available — will use standard attention")

print("\nAll packages installed. Restart the runtime before continuing.")

*Restart the runtime after Cell 0A before continuing.*

In [ ]:
!pip install -q matplotlib tqdm

In [ ]:
# --- Cell 0B: Imports and global configuration ---import os, gc, json, math, time, pickle, random, hashlib, traceback, warnings, re, types, copyimport importlibfrom pathlib import Pathfrom collections import defaultdict, Counterfrom datetime import datetime, timezonefrom typing import Optional, Dict, List, Tuple, Anyimport numpy as npimport pandas as pdimport matplotlibimport matplotlib.pyplot as pltfrom scipy import stats as sp_statsimport torchfrom torch import nnfrom tqdm.auto import tqdmfrom datasets import load_datasetfrom transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, set_seedfrom IPython.display import display# --- Output directory ---# All results (checkpoints, figures, CSVs) are saved here.# Change this to your preferred path.BASE_DIR = os.environ.get("PAPER4_OUTPUT_DIR", "./paper4_results")# --- HuggingFace authentication ---# Some base models may require a HuggingFace token.# Option 1: Set the HF_TOKEN environment variable before running# Option 2: Run `huggingface-cli login` in your terminal# Option 3: Set it here (NOT recommended for public repos)HF_TOKEN = os.environ.get("HF_TOKEN", None)if HF_TOKEN:    print(f"HF Token: configured (***{HF_TOKEN[-4:]})")else:    print("HF Token: not set (set HF_TOKEN env var if models require auth)")CHECKPOINTS_DIR = os.path.join(BASE_DIR, "checkpoints")RESULTS_DIR     = os.path.join(BASE_DIR, "results")FIGURES_DIR     = os.path.join(BASE_DIR, "figures")TABLES_DIR      = os.path.join(BASE_DIR, "tables")LOGS_DIR        = os.path.join(BASE_DIR, "logs")ARTIFACTS_DIR   = os.path.join(BASE_DIR, "artifacts")for d in [BASE_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR, ARTIFACTS_DIR]:    os.makedirs(d, exist_ok=True)DEVICE = "cuda" if torch.cuda.is_available() else "cpu"DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16SEED   = 42set_seed(SEED)VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0print(f"Device: {DEVICE} | VRAM: {VRAM_GB:.1f} GB | dtype: {DTYPE}")# --- Experiment parameters ---USE_BASE_MODELS      = True           # -Base variants (no instruction tuning)PAPER_MODE           = True           # full paper sweep (set False for quick test)NUM_GENERATIONS      = 200 if PAPER_MODE else 50MAX_NEW_TOKENS       = 64SPEEDUP_TRIALS       = 100 if PAPER_MODE else 30SPEEDUP_WARMUP_RUNS  = 5DRAFT_LENGTHS        = [2, 4, 8]TEMPERATURES         = [0.0, 0.6]BOOTSTRAP_N          = 10_000# --- Models to run ---MODELS_CORE = ["qwen3.5-0.8b", "falcon-h1-0.5b", "qwen2.5-0.5b"]MODELS_TO_RUN = MODELS_CORE  # Add "falcon-h1-3b" after running core models# Task promptsTASK_PROMPTS = {    "mmlu":   {"prompts_source": "mmlu",   "n_prompts": 200 if PAPER_MODE else 50},    "gsm8k":  {"prompts_source": "gsm8k",  "n_prompts": 200 if PAPER_MODE else 50},    "alpaca": {"prompts_source": "alpaca",  "n_prompts": 200 if PAPER_MODE else 50},}print(f"\nModels to run: {MODELS_TO_RUN}")print(f"Draft lengths: {DRAFT_LENGTHS} | Temperatures: {TEMPERATURES}")print(f"Generations/condition: {NUM_GENERATIONS} | Speedup trials: {SPEEDUP_TRIALS}")

In [ ]:
# --- Cell 0C: Checkpoint manager ---
class CheckpointManager:
    def __init__(self, base_dir):
        self.base_dir = base_dir
        self.ckpt_dir = os.path.join(base_dir, "checkpoints")
        os.makedirs(self.ckpt_dir, exist_ok=True)
        self.log_file = os.path.join(base_dir, "logs", "experiment.log")
        os.makedirs(os.path.dirname(self.log_file), exist_ok=True)

    def _path(self, name):
        safe = re.sub(r"[^a-zA-Z0-9_\-.]", "_", name)
        return os.path.join(self.ckpt_dir, f"{safe}.pkl")

    def save(self, name, data):
        with open(self._path(name), "wb") as f:
            pickle.dump(data, f)
        self.log(f"Saved: {name}")

    def load(self, name):
        p = self._path(name)
        if os.path.exists(p):
            with open(p, "rb") as f:
                return pickle.load(f)
        return None

    def exists(self, name): return os.path.exists(self._path(name))

    def list_checkpoints(self):
        return [f.replace(".pkl","") for f in os.listdir(self.ckpt_dir) if f.endswith(".pkl")]

    def log(self, msg):
        ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{ts}] {msg}"
        print(line)
        with open(self.log_file, "a") as f:
            f.write(line + "\n")

ckpt = CheckpointManager(BASE_DIR)
ckpt.log("=== Paper 4 v3 session started ===")

def save_figure(fig, name, dpi=300):
    for ext in ["pdf", "png"]:
        fig.savefig(os.path.join(FIGURES_DIR, f"{name}.{ext}"), dpi=dpi, bbox_inches="tight")
    ckpt.log(f"Figure saved: {name}")

def save_dataframe(df, name, **kwargs):
    df.to_csv(os.path.join(RESULTS_DIR, f"{name}.csv"), **kwargs)
    ckpt.log(f"CSV saved: {name}")

def save_latex_table(tex_str, name):
    with open(os.path.join(TABLES_DIR, f"{name}.tex"), "w") as f:
        f.write(tex_str)
    ckpt.log(f"LaTeX table saved: {name}")

def bootstrap_ci(arr, n_boot=BOOTSTRAP_N, ci=0.95):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 3:
        return float(np.mean(arr)), float(np.mean(arr)), float(np.mean(arr))
    means = np.array([np.mean(np.random.choice(arr, len(arr), replace=True)) for _ in range(n_boot)])
    lo = np.percentile(means, (1-ci)/2 * 100)
    hi = np.percentile(means, (1+ci)/2 * 100)
    return float(np.mean(arr)), float(lo), float(hi)

print("Checkpoint manager ready.")

# Section 1 — Model registry

In [ ]:
# --- Cell 1A: Model registry and loading helpers ---MODEL_SPECS = {    "qwen3.5-0.8b": {        "display_name": "Qwen3.5-0.8B",        "model_id": "Qwen/Qwen3.5-0.8B-Base" if USE_BASE_MODELS else "Qwen/Qwen3.5-0.8B",        "arch_family": "qwen_sequential_hybrid", "hybrid_type": "sequential",        "n_layers": 24, "n_alt_layers": 18, "n_attn_layers": 6, "hidden_dim": 1536,    },    "falcon-h1-0.5b": {        "display_name": "Falcon-H1-0.5B",        "model_id": "tiiuae/Falcon-H1-0.5B-Base" if USE_BASE_MODELS else "tiiuae/Falcon-H1-0.5B-Instruct",        "arch_family": "falcon_parallel_hybrid", "hybrid_type": "parallel",        "max_seq_length_no_kernel": 1024,        "n_layers": 36, "n_alt_layers": 36, "n_attn_layers": 36, "hidden_dim": 1024,    },    "qwen2.5-0.5b": {        "display_name": "Qwen2.5-0.5B", "model_id": "Qwen/Qwen2.5-0.5B",        "arch_family": "transformer_baseline", "hybrid_type": "transformer",        "n_layers": 24, "n_alt_layers": 0, "n_attn_layers": 24, "hidden_dim": 896,    },    "falcon-h1-1.5b": {        "display_name": "Falcon-H1-1.5B",        "model_id": "tiiuae/Falcon-H1-1.5B-Base" if USE_BASE_MODELS else "tiiuae/Falcon-H1-1.5B-Instruct",        "arch_family": "falcon_parallel_hybrid", "hybrid_type": "parallel",        "n_layers": 36, "n_alt_layers": 36, "n_attn_layers": 36, "hidden_dim": 1536,    },    "qwen3.5-2b": {        "display_name": "Qwen3.5-2B", "model_id": "Qwen/Qwen3.5-2B-Base",        "arch_family": "qwen_sequential_hybrid", "hybrid_type": "sequential",        "n_layers": 28, "n_alt_layers": 21, "n_attn_layers": 7, "hidden_dim": 1536,    },    "falcon-h1-3b": {        "display_name": "Falcon-H1-3B",        "model_id": "tiiuae/Falcon-H1-3B-Base",        "arch_family": "falcon_parallel_hybrid", "hybrid_type": "parallel",        "n_layers": 32, "n_alt_layers": 32, "n_attn_layers": 32, "hidden_dim": 2048,    },}def get_decoder_layers(model):    for path, fn in [("model.layers", lambda m: getattr(getattr(m,"model",None),"layers",None)),                     ("transformer.h", lambda m: getattr(getattr(m,"transformer",None),"h",None))]:        layers = fn(model)        if isinstance(layers, (nn.ModuleList, list)) and len(layers) > 0: return layers, path    best_name, best_mod = None, None    for name, module in model.named_modules():        if isinstance(module, nn.ModuleList) and len(module) >= 4:            if best_mod is None or len(module) > len(best_mod): best_name, best_mod = name, module    if best_mod is None: raise ValueError("Could not locate decoder layers")    return best_mod, best_namedef get_max_inference_seq_length(model_key):    spec = MODEL_SPECS[model_key]    if "max_seq_length_no_kernel" in spec:        try: import mamba_ssm; return 4096        except ImportError: return spec["max_seq_length_no_kernel"]    return 4096_LOADED_MODELS = {}def load_model_and_tokenizer(model_key):    if model_key in _LOADED_MODELS: return _LOADED_MODELS[model_key]    spec = MODEL_SPECS[model_key]    ckpt.log(f"Loading {model_key} from {spec['model_id']}")    tok_kw = dict(trust_remote_code=True, use_fast=True)    if HF_TOKEN: tok_kw["token"] = HF_TOKEN    tokenizer = AutoTokenizer.from_pretrained(spec["model_id"], **tok_kw)    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token    tokenizer.padding_side = "left"    kw = dict(torch_dtype=DTYPE, device_map="auto", trust_remote_code=True)    if HF_TOKEN: kw["token"] = HF_TOKEN    try:        model = AutoModelForCausalLM.from_pretrained(spec["model_id"], attn_implementation="flash_attention_2", **kw)        ckpt.log("  Using Flash Attention 2")    except torch.cuda.OutOfMemoryError:        gc.collect(); torch.cuda.empty_cache()        raise RuntimeError(f"OOM loading {model_key} — try unloading other models first or use a larger GPU")    except Exception:        try:            model = AutoModelForCausalLM.from_pretrained(spec["model_id"], **kw)        except torch.cuda.OutOfMemoryError:            gc.collect(); torch.cuda.empty_cache()            raise RuntimeError(f"OOM loading {model_key} — insufficient VRAM ({VRAM_GB:.1f}GB)")    model.eval()    # Discover actual layer types for Qwen hybrids    if spec["arch_family"] == "qwen_sequential_hybrid":        lt = getattr(model.config, "layer_types", None)        if lt:            spec["n_alt_layers"] = sum(1 for t in lt if t in ("linear_attention","linear"))            spec["n_attn_layers"] = sum(1 for t in lt if t in ("full_attention","attention"))            spec["n_layers"] = spec["n_alt_layers"] + spec["n_attn_layers"]    # Extract real dimensions from config for FLOP estimation    cfg = model.config    spec["_intermediate_size"] = getattr(cfg, "intermediate_size", spec["hidden_dim"] * 4)    spec["_num_attention_heads"] = getattr(cfg, "num_attention_heads", 16)    spec["_num_kv_heads"] = getattr(cfg, "num_key_value_heads", spec.get("_num_attention_heads", 16))    spec["_head_dim"] = spec["hidden_dim"] // spec["_num_attention_heads"]    _LOADED_MODELS[model_key] = (model, tokenizer)    ckpt.log(f"  {model_key}: {sum(p.numel() for p in model.parameters()):,} params")    return model, tokenizerdef unload_model(model_key):    if model_key in _LOADED_MODELS: del _LOADED_MODELS[model_key]    gc.collect()    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Section 2 — Draft model construction (v3: cache-compatible, truly zero-cost)

**Key change from v2:** Skip functions now return cache-compatible outputs so `use_cache=True` works with patched models. This enables KV-cached draft generation, which is critical for honest speedup measurements.

The skip functions:
- **Falcon SSM-only:** Attention `forward()` → returns `(zeros, dummy_kv_cache, None)` without executing QKV projections
- **Qwen Linear-only:** Full attention layer `forward()` → returns `(identity, None, None, None)` 
- **LayerSkip (33%):** Skips middle 33% of layers (was 50% in v2)
- **Early-exit (new):** First 50% of layers, tested on hybrid models as comparison

In [ ]:
# --- Cell 2A: DraftModelManager — cache-compatible, computation-skipping ---

def normalize_qwen_layer_type(lt):
    return {"linear_attention":"linear","full_attention":"attention","linear":"linear","attention":"attention"}.get(lt, lt)

def _find_attention_submodule(layer):
    """Find the attention submodule within a decoder layer."""
    if hasattr(layer, "self_attn"): return layer.self_attn, "self_attn"
    for name, mod in layer.named_children():
        cn = mod.__class__.__name__.lower()
        if "attention" in cn or "attn" in cn: return mod, name
    return None, None

def _make_cache_compatible_zero_attn(attn_mod, model_config):
    """Create a skip function that returns (zeros, valid_cache, None).
    
    The cache entry must be valid so model.generate(use_cache=True) works.
    We create minimal KV tensors of the right shape so the cache infrastructure
    doesn't crash, but the values are all zeros (never actually used).
    """
    n_kv_heads = getattr(model_config, "num_key_value_heads",
                  getattr(model_config, "num_attention_heads", 1))
    head_dim = getattr(model_config, "hidden_size", 1024) // getattr(model_config, "num_attention_heads", 1)
    orig_forward = attn_mod.forward

    def skip_forward(*args, **kwargs):
        # Extract hidden_states from args or kwargs
        h = args[0] if args else kwargs.get("hidden_states")
        if h is None:
            for a in args:
                if torch.is_tensor(a) and a.dim() >= 2: h = a; break
        if h is None:
            raise ValueError("Could not find hidden_states in attention forward")

        bsz, seq_len = h.shape[0], h.shape[1]
        zero_out = torch.zeros_like(h)

        use_cache = kwargs.get("use_cache", False)
        if use_cache:
            # Create dummy KV cache entries of the correct shape
            # Shape: (batch, n_kv_heads, seq_len, head_dim)
            dummy_k = torch.zeros(bsz, n_kv_heads, seq_len, head_dim,
                                  device=h.device, dtype=h.dtype)
            dummy_v = torch.zeros_like(dummy_k)

            # Handle past_key_values (append to existing cache if present)
            past = kwargs.get("past_key_values", None)
            if past is not None:
                # Try DynamicCache API first
                try:
                    layer_idx = kwargs.get("layer_idx", kwargs.get("cache_position", None))
                    # For DynamicCache, we need to update it properly
                    if hasattr(past, "update"):
                        past.update(dummy_k, dummy_v, layer_idx if isinstance(layer_idx, int) else 0)
                        return (zero_out, past, None)
                except Exception:
                    pass
                # For tuple-based cache, just return current KV
                return (zero_out, (dummy_k, dummy_v), None)
            return (zero_out, (dummy_k, dummy_v), None)
        return (zero_out, None, None)

    skip_forward._is_draft_skip = True
    return skip_forward, orig_forward

def _make_identity_layer_forward(layer):
    """Replace full layer forward with identity (for Qwen attention layers or LayerSkip)."""
    orig_forward = layer.forward

    def identity_forward(*args, **kwargs):
        h = args[0] if args else kwargs.get("hidden_states")
        if h is None:
            for a in args:
                if torch.is_tensor(a) and a.dim() >= 2: h = a; break
        # Return h unchanged + None for other expected outputs
        # Most HF layers return (hidden_states, present_kv, attentions, ...)
        return (h,) + (None,) * 3

    identity_forward._is_draft_skip = True
    return identity_forward, orig_forward


class DraftModelManager:
    """Manages transformation of hybrid model into SSM-only / linear-only draft subgraph.

    v3 changes:
    - Skip functions return cache-compatible outputs (valid KV entries)
    - This enables model.generate(use_cache=True) with patched models
    - LayerSkip reduced from 50% to 33%
    - Added early-exit strategy for hybrid model comparison
    """
    def __init__(self, model, model_key):
        self.model, self.model_key = model, model_key
        self.spec = MODEL_SPECS[model_key]
        self.patched_forwards = {}
        self.active = False
        self._strategy = None

    @property
    def strategy(self):
        return self._strategy

    def activate_draft_mode(self, strategy="ssm_only"):
        if self.active: self.deactivate_draft_mode()
        arch = self.spec["arch_family"]
        if strategy == "ssm_only" and arch == "falcon_parallel_hybrid":
            self._falcon_ssm_only()
        elif strategy == "linear_only" and arch == "qwen_sequential_hybrid":
            self._qwen_linear_only()
        elif strategy == "layer_skip":
            self._layer_skip()
        elif strategy == "early_exit":
            self._early_exit()
        else:
            raise ValueError(f"Strategy '{strategy}' not supported for {arch}")
        self.active = True
        self._strategy = strategy

    def deactivate_draft_mode(self):
        for key, item in list(self.patched_forwards.items()):
            item["module"].forward = item["original"]
        self.patched_forwards.clear()
        self.active = False
        self._strategy = None

    def __enter__(self): return self
    def __exit__(self, *args): self.deactivate_draft_mode()

    # --- Falcon: SKIP attention computation, return valid cache ---
    def _falcon_ssm_only(self):
        layers, _ = get_decoder_layers(self.model)
        n_patched = 0
        for idx, layer in enumerate(layers):
            attn_mod, _ = _find_attention_submodule(layer)
            if attn_mod is None: continue
            skip_fwd, orig_fwd = _make_cache_compatible_zero_attn(attn_mod, self.model.config)
            attn_mod.forward = skip_fwd
            self.patched_forwards[f"f_attn_{idx}"] = {"module": attn_mod, "original": orig_fwd}
            n_patched += 1
        ckpt.log(f"Falcon SSM-only: replaced attn forward in {n_patched}/{len(layers)} layers (cache-compatible)")

    # --- Qwen: skip full attention layers (identity) ---
    def _qwen_linear_only(self):
        layers, _ = get_decoder_layers(self.model)
        lt_raw = getattr(self.model.config, "layer_types", None)
        if lt_raw is None: raise ValueError("Missing layer_types in config")
        lt = [normalize_qwen_layer_type(t) for t in lt_raw]
        skipped = 0
        for idx, layer in enumerate(layers):
            if idx < len(lt) and lt[idx] == "attention":
                id_fwd, orig_fwd = _make_identity_layer_forward(layer)
                layer.forward = id_fwd
                self.patched_forwards[f"q_attn_{idx}"] = {"module": layer, "original": orig_fwd}
                skipped += 1
        ckpt.log(f"Qwen linear-only: skipped {skipped} attn layers (identity forward)")

    # --- LayerSkip baseline (33% middle layers) ---
    def _layer_skip(self, skip_ratio=0.33):
        layers, _ = get_decoder_layers(self.model)
        n = len(layers)
        ns = max(1, int(n * skip_ratio))
        mid = n // 2
        skip_start = mid - ns // 2
        for idx in range(skip_start, skip_start + ns):
            if idx >= n: break
            layer = layers[idx]
            id_fwd, orig_fwd = _make_identity_layer_forward(layer)
            layer.forward = id_fwd
            self.patched_forwards[f"skip_{idx}"] = {"module": layer, "original": orig_fwd}
        ckpt.log(f"LayerSkip: skipped {ns} of {n} layers (ratio={skip_ratio:.0%}, indices {skip_start}-{skip_start+ns-1})")

    # --- Early exit (first E layers only) ---
    def _early_exit(self, exit_frac=0.5):
        layers, _ = get_decoder_layers(self.model)
        n = len(layers)
        el = max(1, int(n * exit_frac))
        for idx in range(el, n):
            layer = layers[idx]
            id_fwd, orig_fwd = _make_identity_layer_forward(layer)
            layer.forward = id_fwd
            self.patched_forwards[f"ee_{idx}"] = {"module": layer, "original": orig_fwd}
        ckpt.log(f"Early-exit: after layer {el} of {n}")

    # --- FLOP ratio for theoretical speedup (uses real config dimensions) ---
    @staticmethod
    def estimate_flop_ratio(model_key, seq_len=256):
        """Estimate c_draft / c_full FLOP ratio using actual model dimensions.

        Uses:
        - Attention FLOPs: 2*d*d_kv*n_q/n_kv (QKV proj) + 2*seq*d (dot products) + 2*d*d (output proj)
        - SSM/Linear FLOPs: ~8*d*d (state expansion + gates + output) [Mamba-2 / linear attn]
        - MLP FLOPs: 3*d*d_ff (gate + up + down for SwiGLU)
        """
        spec = MODEL_SPECS[model_key]
        d = spec["hidden_dim"]
        L = spec["n_layers"]
        na = spec["n_alt_layers"]      # SSM/linear layers
        nat = spec["n_attn_layers"]    # attention layers
        d_ff = spec.get("_intermediate_size", d * 4)
        n_heads = spec.get("_num_attention_heads", 16)
        n_kv = spec.get("_num_kv_heads", n_heads)
        head_d = spec.get("_head_dim", d // n_heads)

        # Per-token FLOPs for each component
        # QKV: 2*d*(n_heads*head_d + 2*n_kv*head_d) for Q, K, V projections
        f_qkv = 2 * d * (n_heads * head_d + 2 * n_kv * head_d)
        # Attention dot products: 2 * n_heads * seq_len * head_d (QK^T + attn@V)
        f_attn_dot = 2 * n_heads * seq_len * head_d
        # Output projection: 2*d*d
        f_attn_out = 2 * d * d
        f_attn_total = f_qkv + f_attn_dot + f_attn_out

        # SSM/Linear attention: ~8*d*d (expand + gate + contract)
        f_ssm = 8 * d * d

        # MLP (SwiGLU): 3 * 2 * d * d_ff (gate, up, down — each is a matmul)
        f_mlp = 3 * 2 * d * d_ff

        if spec["hybrid_type"] == "parallel":
            # Falcon: each layer has SSM + attention + MLP in parallel
            c_full = L * (f_ssm + f_attn_total + f_mlp)
            c_draft = L * (f_ssm + f_mlp)  # skip attention entirely
        elif spec["hybrid_type"] == "sequential":
            # Qwen: some layers are linear-attn, some are full-attn
            c_full = na * (f_ssm + f_mlp) + nat * (f_attn_total + f_mlp)
            c_draft = na * (f_ssm + f_mlp)  # skip attention layers entirely
        else:
            # Transformer: LayerSkip/early-exit
            c_full = L * (f_attn_total + f_mlp)
            c_draft = c_full * 0.67  # 33% LayerSkip
        return c_draft / c_full if c_full > 0 else 1.0

    # --- Compute draft/full FLOPs in GFLOPs for reporting ---
    @staticmethod
    def estimate_flops_gflops(model_key, seq_len=256):
        spec = MODEL_SPECS[model_key]; d = spec["hidden_dim"]; L = spec["n_layers"]
        na, nat = spec["n_alt_layers"], spec["n_attn_layers"]
        d_ff = spec.get("_intermediate_size", d * 4)
        n_heads = spec.get("_num_attention_heads", 16)
        n_kv = spec.get("_num_kv_heads", n_heads)
        head_d = spec.get("_head_dim", d // n_heads)
        f_attn = 2*d*(n_heads*head_d + 2*n_kv*head_d) + 2*n_heads*seq_len*head_d + 2*d*d
        f_ssm = 8*d*d; f_mlp = 6*d*d_ff
        if spec["hybrid_type"] == "parallel":
            return L*(f_ssm+f_mlp)/1e9, L*(f_ssm+f_attn+f_mlp)/1e9
        elif spec["hybrid_type"] == "sequential":
            return na*(f_ssm+f_mlp)/1e9, (na*(f_ssm+f_mlp)+nat*(f_attn+f_mlp))/1e9
        else:
            return L*(f_attn+f_mlp)*0.67/1e9, L*(f_attn+f_mlp)/1e9

print("DraftModelManager ready (v3: cache-compatible).")

In [ ]:
# --- HOTFIX: match exact return formats for Qwen3.5 and Falcon-H1 ---

# Fix 1: Qwen3.5 layers return bare tensor when use_cache=False
def _make_identity_layer_forward(layer):
    orig_forward = layer.forward
    def identity_forward(*args, **kwargs):
        h = args[0] if args else kwargs.get("hidden_states")
        if h is None:
            for a in args:
                if torch.is_tensor(a) and a.dim() >= 2: h = a; break
        use_cache = kwargs.get("use_cache", False)
        if use_cache:
            return (h, None)
        return h
    identity_forward._is_draft_skip = True
    return identity_forward, orig_forward

# Fix 2: Falcon self_attn expects exactly 2 return values
def _make_cache_compatible_zero_attn(attn_mod, model_config):
    orig_forward = attn_mod.forward
    def skip_forward(*args, **kwargs):
        h = args[0] if args else kwargs.get("hidden_states")
        if h is None:
            for a in args:
                if torch.is_tensor(a) and a.dim() >= 2: h = a; break
        zero_out = torch.zeros_like(h)
        return (zero_out, None)
    skip_forward._is_draft_skip = True
    return skip_forward, orig_forward

print("Hotfix applied ✓")

In [ ]:
# --- Cell 2B: Smoke test — verify draft is valid AND faster ---
def smoke_test_draft(model_key):
    """Verify that:
    1. Draft model produces different but related logits (D_TV > 0)
    2. Draft forward is actually faster than full forward (the whole point)
    3. model.generate() works with patched model + use_cache=True
    """
    model, tokenizer = load_model_and_tokenizer(model_key)
    prompt = "The capital of France is"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    inp_len = inputs["input_ids"].shape[1]

    # Full model logits
    with torch.inference_mode():
        fl = model(**inputs, use_cache=False).logits[:, -1, :].float()
    ft5 = torch.topk(fl, 5).indices[0].tolist()
    fp = torch.softmax(fl, dim=-1)

    # Determine strategy
    strategy = ("ssm_only" if MODEL_SPECS[model_key]["arch_family"]=="falcon_parallel_hybrid"
                else "linear_only" if MODEL_SPECS[model_key]["arch_family"]=="qwen_sequential_hybrid"
                else "layer_skip")

    # Draft model logits
    drafter = DraftModelManager(model, model_key)
    drafter.activate_draft_mode(strategy)
    with torch.inference_mode():
        dl = model(**inputs, use_cache=False).logits[:, -1, :].float()
    dt5 = torch.topk(dl, 5).indices[0].tolist()
    dp = torch.softmax(dl, dim=-1)

    d_tv = 0.5 * torch.sum(torch.abs(fp - dp)).item()
    top1_match = ft5[0] == dt5[0]

    # Test model.generate() with cache on patched model
    generate_cache_ok = False
    try:
        with torch.inference_mode():
            gen_out = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                                     use_cache=True, output_scores=True,
                                     return_dict_in_generate=True)
        generate_cache_ok = True
    except Exception as e:
        ckpt.log(f"  WARNING: model.generate(use_cache=True) failed for {model_key}: {e}")
    drafter.deactivate_draft_mode()

    # Timing comparison: full vs draft forward
    # Warmup
    for _ in range(SPEEDUP_WARMUP_RUNS):
        with torch.inference_mode(): _ = model(**inputs, use_cache=False)
    torch.cuda.synchronize()

    N = 30
    se = torch.cuda.Event(enable_timing=True); ee = torch.cuda.Event(enable_timing=True)
    se.record()
    for _ in range(N):
        with torch.inference_mode(): _ = model(**inputs, use_cache=False)
    ee.record(); torch.cuda.synchronize()
    full_ms = se.elapsed_time(ee) / N

    drafter.activate_draft_mode(strategy)
    # Warmup draft
    for _ in range(3):
        with torch.inference_mode(): _ = model(**inputs, use_cache=False)
    torch.cuda.synchronize()
    se.record()
    for _ in range(N):
        with torch.inference_mode(): _ = model(**inputs, use_cache=False)
    ee.record(); torch.cuda.synchronize()
    draft_ms = se.elapsed_time(ee) / N
    drafter.deactivate_draft_mode()

    speedup_ratio = full_ms / draft_ms if draft_ms > 0 else float("inf")
    draft_gf, full_gf = DraftModelManager.estimate_flops_gflops(model_key)

    print(f"\n{'='*75}")
    print(f"SMOKE TEST: {model_key} | Strategy: {strategy}")
    print(f"  Full  top-5: {[tokenizer.decode(t) for t in ft5]}")
    print(f"  Draft top-5: {[tokenizer.decode(t) for t in dt5]}")
    print(f"  D_TV = {d_tv:.4f} | Top-1 match: {top1_match}")
    print(f"  Full={full_ms:.2f}ms | Draft={draft_ms:.2f}ms | Draft/Full={draft_ms/full_ms:.3f} | Speedup={speedup_ratio:.2f}x")
    print(f"  Theoretical FLOPs: draft={draft_gf:.2f} GF, full={full_gf:.2f} GF, ratio={draft_gf/full_gf:.3f}")
    print(f"  model.generate(use_cache=True): {'OK' if generate_cache_ok else 'FAILED (will use fallback)'}")
    if speedup_ratio < 1.05:
        print(f"  ⚠ WARNING: Draft is NOT meaningfully faster than full model!")
    print(f"{'='*75}")

    return {"model_key": model_key, "strategy": strategy, "d_tv": d_tv,
            "full_ms": full_ms, "draft_ms": draft_ms, "speedup_ratio": speedup_ratio,
            "top1_match": top1_match, "generate_cache_ok": generate_cache_ok}

smoke_results = {}
for mk in MODELS_TO_RUN:
    try: smoke_results[mk] = smoke_test_draft(mk)
    except Exception as e: print(f"[ERROR] {mk}: {e}"); traceback.print_exc()
    unload_model(mk)

In [ ]:
# Just the function definition - paste from Cell A hotfix
def measure_speedup_v2(model_key, k, temperature=0.0, n_trials=None, max_new_tokens=None):
    if n_trials is None: n_trials = SPEEDUP_TRIALS
    if max_new_tokens is None: max_new_tokens = MAX_NEW_TOKENS
    model, tokenizer = load_model_and_tokenizer(model_key)
    arch = MODEL_SPECS[model_key]["arch_family"]
    strategy = ("ssm_only" if arch == "falcon_parallel_hybrid"
                else "linear_only" if arch == "qwen_sequential_hybrid" else "layer_skip")
    mx = get_max_inference_seq_length(model_key)
    mp = min(256, mx - max_new_tokens - 20)
    ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")
    texts = [e["text"].strip() for e in ds if len(e["text"].strip()) > 50][:n_trials]
    drafter = DraftModelManager(model, model_key)
    wi = tokenizer(texts[0], return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)
    for _ in range(SPEEDUP_WARMUP_RUNS):
        with torch.inference_mode():
            try:
                model.generate(**wi, max_new_tokens=16, do_sample=False, use_cache=True)
            except Exception:
                _ = model(**wi, use_cache=False)
    torch.cuda.synchronize()
    ar_times, ar_tokens = [], []
    for p in tqdm(texts[:n_trials], desc=f"{model_key} AR baseline"):
        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)
        se = torch.cuda.Event(enable_timing=True); ee = torch.cuda.Event(enable_timing=True)
        se.record()
        with torch.inference_mode():
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=(temperature > 0),
                                 temperature=max(temperature, 1e-6) if temperature > 0 else None, use_cache=True)
        ee.record(); torch.cuda.synchronize()
        ar_times.append(se.elapsed_time(ee)); ar_tokens.append(out.shape[1] - inp["input_ids"].shape[1])
    ar_tps = sum(ar_tokens) / (sum(ar_times) / 1000)
    sp_times, sp_tokens = [], []
    for p in tqdm(texts[:n_trials], desc=f"{model_key} Spec k={k}"):
        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)
        cids = inp["input_ids"]; gen = 0
        se = torch.cuda.Event(enable_timing=True); ee = torch.cuda.Event(enable_timing=True)
        se.record()
        with torch.inference_mode():
            while gen < max_new_tokens:
                drafter.activate_draft_mode(strategy)
                draft_tokens, draft_probs = [], []
                draft_past, draft_input = None, cids
                for step in range(k):
                    try:
                        out = model(input_ids=draft_input, past_key_values=draft_past, use_cache=True)
                        draft_past = out.past_key_values
                    except Exception:
                        full_seq = torch.cat([cids] + [torch.tensor([[t]], device=DEVICE) for t in draft_tokens], dim=-1) if draft_tokens else cids
                        out = model(input_ids=full_seq, use_cache=False)
                        draft_past = None
                    lg = out.logits[:, -1, :].float(); pr = torch.softmax(lg, dim=-1)
                    tk = lg.argmax(dim=-1, keepdim=True) if temperature == 0.0 else torch.multinomial(torch.softmax(lg / temperature, dim=-1), 1)
                    draft_tokens.append(tk.item()); draft_probs.append(pr[0, tk.item()].item())
                    draft_input = tk
                del draft_past; drafter.deactivate_draft_mode()
                actual_k = len(draft_tokens)
                if actual_k == 0: break
                dt_tensor = torch.tensor([draft_tokens], device=DEVICE)
                vi = torch.cat([cids, dt_tensor], dim=-1)
                vo = model(input_ids=vi, use_cache=False)
                fl = vo.logits[:, cids.shape[1]-1 : cids.shape[1]-1+actual_k, :].float()
                na = 0
                for i in range(actual_k):
                    if temperature == 0.0:
                        if fl[:, i, :].argmax(dim=-1).item() == draft_tokens[i]: na += 1
                        else: break
                    else:
                        fp = torch.softmax(fl[:, i, :] / temperature, dim=-1)
                        if draft_probs[i] > 0 and random.random() < min(1.0, fp[0, draft_tokens[i]].item() / draft_probs[i]): na += 1
                        else: break
                if na < actual_k:
                    bonus = fl[:, na, :].argmax(dim=-1, keepdim=True) if temperature == 0.0 else torch.multinomial(torch.softmax(fl[:, na, :] / temperature, dim=-1), 1)
                    acc = draft_tokens[:na] + [bonus.item()]
                else:
                    ll = vo.logits[:, -1, :].float()
                    bonus = ll.argmax(dim=-1, keepdim=True) if temperature == 0.0 else torch.multinomial(torch.softmax(ll / temperature, dim=-1), 1)
                    acc = draft_tokens + [bonus.item()]
                cids = torch.cat([cids, torch.tensor([acc], device=DEVICE)], dim=-1); gen += len(acc)
        ee.record(); torch.cuda.synchronize()
        sp_times.append(se.elapsed_time(ee)); sp_tokens.append(gen)
    sp_tps = sum(sp_tokens) / (sum(sp_times) / 1000)
    return {"model_key": model_key, "strategy": strategy, "k": k, "temperature": temperature,
            "ar_tok_per_sec": ar_tps, "spec_tok_per_sec": sp_tps,
            "speedup": sp_tps / ar_tps if ar_tps > 0 else 0, "n_trials": n_trials}
print("measure_speedup_v2 defined ✓")

In [ ]:
# --- Falcon-H1-3B experiments ---# Falcon-H1-3B is registered in MODEL_SPECS (Cell 1A)# --- Falcon-H1-3B: all experiments in one cell ---mk = "falcon-h1-3b"# === Divergence ===model, tokenizer = load_model_and_tokenizer(mk)drafter = DraftModelManager(model, mk)ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")texts = [e["text"].strip() for e in ds if len(e["text"].strip())>200][:50]mx = get_max_inference_seq_length(mk)all_dtv, all_top1 = [], []for text in tqdm(texts, desc=f"{mk} divergence"):    ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=min(138, mx)).to(DEVICE)["input_ids"]    with torch.inference_mode():        fl = model(input_ids=ids, use_cache=False).logits[:, :-1, :].float()    drafter.activate_draft_mode("ssm_only")    with torch.inference_mode():        dl = model(input_ids=ids, use_cache=False).logits[:, :-1, :].float()    drafter.deactivate_draft_mode()    fp, dp = torch.softmax(fl, dim=-1), torch.softmax(dl, dim=-1)    all_dtv.append((0.5*torch.sum(torch.abs(fp-dp), dim=-1)).squeeze(0).cpu().numpy())    all_top1.append((fl.argmax(dim=-1)==dl.argmax(dim=-1)).float().squeeze(0).cpu().numpy())dtv_flat = np.concatenate(all_dtv)div_result = {"model_key": mk, "d_tv_mean": float(np.mean(dtv_flat)), "d_tv_std": float(np.std(dtv_flat)),              "top1_agreement": float(np.mean(np.concatenate(all_top1)))}ckpt.save(f"divergence_v3__{mk}", div_result)print(f"D_TV={div_result['d_tv_mean']:.4f}, top1={div_result['top1_agreement']:.3f}")unload_model(mk)# === Acceptance rate ===model, tokenizer = load_model_and_tokenizer(mk)drafter = DraftModelManager(model, mk)prompts = [e["text"].strip() for e in ds if len(e["text"].strip())>100][:200]for k in [2, 4, 8]:    for temp in [0.0, 0.6]:        en = f"acceptance_v3__{mk}__k{k}__T{temp}"        if ckpt.load(en): print(f"  {en} cached"); continue        accepted = []        for ti in tqdm(range(min(200, len(prompts))), desc=f"{mk} k={k} T={temp}"):            inp = tokenizer(prompts[ti], return_tensors="pt", truncation=True, max_length=min(512, mx-k-10)).to(DEVICE)            iids = inp["input_ids"]            drafter.activate_draft_mode("ssm_only")            dt, dp, cids = [], [], iids.clone()            with torch.inference_mode():                for _ in range(k):                    o = model(input_ids=cids, use_cache=False)                    lg = o.logits[:,-1,:].float()                    pr = torch.softmax(lg, dim=-1) if temp==0.0 else torch.softmax(lg/temp, dim=-1)                    tk = lg.argmax(dim=-1, keepdim=True) if temp==0.0 else torch.multinomial(pr, 1)                    dt.append(tk.item()); dp.append(pr[0,tk.item()].item())                    cids = torch.cat([cids, tk], dim=-1)            drafter.deactivate_draft_mode()            vid = torch.cat([iids, torch.tensor([dt], device=DEVICE)], dim=-1)            with torch.inference_mode():                fl = model(input_ids=vid, use_cache=False).logits[:, iids.shape[1]-1:iids.shape[1]-1+k, :].float()            na = 0            for i in range(k):                if temp==0.0:                    if fl[:,i,:].argmax(dim=-1).item()==dt[i]: na+=1                    else: break                else:                    pf = torch.softmax(fl[:,i,:]/temp, dim=-1)[0,dt[i]].item()                    if dp[i]>0 and random.random()<min(1.0,pf/dp[i]): na+=1                    else: break            accepted.append(na)        arr = np.array(accepted)/k        mr, cl, ch = bootstrap_ci(arr)        r = {"model_key":mk,"strategy":"ssm_only","k":k,"temperature":temp,             "acceptance_rate":mr,"acceptance_rate_ci_lo":cl,"acceptance_rate_ci_hi":ch,             "mean_accepted_length":float(np.mean(accepted)),"n_trials":len(accepted)}        ckpt.save(en, r)        print(f"  k={k} T={temp}: α={mr:.3f} [{cl:.3f},{ch:.3f}]")unload_model(mk)# === Speedup ===model, tokenizer = load_model_and_tokenizer(mk)for k in [2, 4, 8]:    en = f"speedup_v3fix__{mk}__k{k}"    if ckpt.load(en): print(f"  {en} cached"); continue    r = measure_speedup_v2(mk, k)    ckpt.save(en, r)    print(f"  k={k}: {r['speedup']:.3f}x")unload_model(mk)print("\n=== Falcon-H1-3B COMPLETE ===")

# Section 3 — Experiment 1: Token-level logit divergence ($D_{TV}$)
Measures $D_{TV}(P_{draft}, P_{full})$ across sequence positions on WikiText-2.

Produces: `divergence_summary.csv`, Figure 1.

In [ ]:
# --- Cell 3A: Divergence measurement ---
def measure_divergence(model_key, n_seqs=50, chunk_size=128):
    """Measure D_TV between draft and full model logits position-by-position."""
    model, tokenizer = load_model_and_tokenizer(model_key)
    arch = MODEL_SPECS[model_key]["arch_family"]
    strategy = ("ssm_only" if arch=="falcon_parallel_hybrid"
                else "linear_only" if arch=="qwen_sequential_hybrid" else "layer_skip")
    mx = get_max_inference_seq_length(model_key)

    ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")
    texts = [e["text"].strip() for e in ds if len(e["text"].strip())>200][:n_seqs]

    all_dtv = []; all_top1 = []; all_kl = []
    drafter = DraftModelManager(model, model_key)

    for text in tqdm(texts, desc=f"{model_key} divergence"):
        ids = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=min(chunk_size + 10, mx)).to(DEVICE)["input_ids"]
        s, e = 0, min(ids.shape[1], chunk_size)
        with torch.inference_mode():
            full_logits = model(input_ids=ids[:, :e], use_cache=False).logits[:, s:e-1, :].float()
        drafter.activate_draft_mode(strategy)
        with torch.inference_mode():
            draft_logits = model(input_ids=ids[:, :e], use_cache=False).logits[:, s:e-1, :].float()
        drafter.deactivate_draft_mode()

        fp = torch.softmax(full_logits, dim=-1)
        dp = torch.softmax(draft_logits, dim=-1)
        # D_TV per position
        dtv = 0.5 * torch.sum(torch.abs(fp - dp), dim=-1).squeeze(0)
        # Top-1 agreement per position
        t1 = (full_logits.argmax(dim=-1) == draft_logits.argmax(dim=-1)).float().squeeze(0)
        # KL divergence per position (draft || full)
        kl = torch.sum(fp * (torch.log(fp + 1e-10) - torch.log(dp + 1e-10)), dim=-1).squeeze(0)

        all_dtv.append(dtv.cpu().numpy())
        all_top1.append(t1.cpu().numpy())
        all_kl.append(kl.cpu().numpy())

    dtv_flat = np.concatenate(all_dtv)
    t1_flat = np.concatenate(all_top1)
    kl_flat = np.concatenate(all_kl)

    return {"model_key": model_key, "strategy": strategy,
            "d_tv_mean": float(np.mean(dtv_flat)), "d_tv_std": float(np.std(dtv_flat)),
            "d_tv_median": float(np.median(dtv_flat)),
            "d_tv_p90": float(np.percentile(dtv_flat, 90)),
            "top1_agreement": float(np.mean(t1_flat)),
            "kl_mean": float(np.mean(kl_flat)), "kl_std": float(np.std(kl_flat)),
            "n_positions": len(dtv_flat), "n_seqs": len(texts),
            "dtv_per_position": dtv_flat[:500].tolist()}  # first 500 for plotting

In [ ]:
# --- Cell 3B: Run divergence + Figure 1 ---
divergence_results = {}
for mk in MODELS_TO_RUN:
    en = f"divergence_v3__{mk}"
    cached = ckpt.load(en)
    if cached is not None and "d_tv_mean" in cached:
        ckpt.log(f"  {en} from cache"); divergence_results[mk] = cached; continue
    try:
        r = measure_divergence(mk)
        ckpt.save(en, r); divergence_results[mk] = r
    except Exception as e: ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)

# Display summary
if divergence_results:
    drows = [{"model": mk, "D_TV_mean": r["d_tv_mean"], "D_TV_std": r["d_tv_std"],
              "D_TV_median": r["d_tv_median"], "D_TV_p90": r["d_tv_p90"],
              "top1_agree": r["top1_agreement"], "KL_mean": r["kl_mean"],
              "n_pos": r["n_positions"]}
             for mk, r in divergence_results.items()]
    ddf = pd.DataFrame(drows)
    save_dataframe(ddf, "divergence_summary", index=False)
    display(ddf)

    # Figure 1: D_TV distribution + top-1 agreement
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    for mk, r in divergence_results.items():
        vals = r.get("dtv_per_position", [])
        if vals:
            lb = MODEL_SPECS[mk]["display_name"]
            a1.hist(vals, bins=50, alpha=0.5, label=lb, density=True)
    a1.set_xlabel("$D_{TV}(P_{draft}, P_{full})$"); a1.set_ylabel("Density")
    a1.set_title("(a) Distribution of token-level divergence"); a1.legend(); a1.grid(True, alpha=0.3)

    models = list(divergence_results.keys())
    x = np.arange(len(models))
    a2.bar(x - 0.15, [divergence_results[m]["d_tv_mean"] for m in models], 0.3, label="D_TV mean", alpha=0.8)
    a2.bar(x + 0.15, [1 - divergence_results[m]["top1_agreement"] for m in models], 0.3,
           label="1 - top1 agree", alpha=0.8)
    a2.set_xticks(x); a2.set_xticklabels([MODEL_SPECS[m]["display_name"] for m in models], rotation=15)
    a2.set_ylabel("Value"); a2.set_title("(b) D_TV vs top-1 disagreement")
    a2.legend(); a2.grid(True, alpha=0.3, axis="y")
    fig.suptitle("Exp 1: Token-level logit divergence between draft and full model", fontsize=12)
    fig.tight_layout(); save_figure(fig, "paper_divergence")

# Section 4 — Experiment 2: Acceptance rate ($\alpha$)
Simulates speculative decoding acceptance (Leviathan et al. rejection sampling).

Produces: `acceptance_rate.csv`, Figure 2.

In [ ]:
# --- Cell 4A: Acceptance rate simulator (v3: cached verify) ---
def simulate_acceptance_rate(model_key, k, temperature=0.0, n_trials=None, prompts=None):
    """Simulate speculative decoding and measure acceptance rate.
    
    v3: Verify phase uses single forward pass (already efficient).
         Draft phase uses use_cache=False (acceptable: we're measuring α, not timing).
    """
    if n_trials is None: n_trials = NUM_GENERATIONS
    if prompts is None:
        ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")
        prompts = [e["text"].strip() for e in ds if len(e["text"].strip())>100][:n_trials]
    model, tokenizer = load_model_and_tokenizer(model_key)
    arch = MODEL_SPECS[model_key]["arch_family"]
    strategy = ("ssm_only" if arch=="falcon_parallel_hybrid"
                else "linear_only" if arch=="qwen_sequential_hybrid" else "layer_skip")
    mx = get_max_inference_seq_length(model_key)

    accepted_lengths = []
    drafter = DraftModelManager(model, model_key)

    for ti in tqdm(range(min(n_trials, len(prompts))), desc=f"{model_key} k={k} T={temperature}"):
        inp = tokenizer(prompts[ti], return_tensors="pt", truncation=True,
                        max_length=min(512, mx-k-10)).to(DEVICE)
        iids = inp["input_ids"]

        # --- Draft phase: generate k tokens with patched model ---
        drafter.activate_draft_mode(strategy)
        draft_tokens = []; draft_probs = []; cids = iids.clone()
        with torch.inference_mode():
            for _ in range(k):
                o = model(input_ids=cids, use_cache=False)
                lg = o.logits[:, -1, :].float()
                if temperature == 0.0:
                    pr = torch.softmax(lg, dim=-1)
                    tk = lg.argmax(dim=-1, keepdim=True)
                else:
                    pr = torch.softmax(lg / temperature, dim=-1)
                    tk = torch.multinomial(pr, 1)
                draft_tokens.append(tk.item())
                draft_probs.append(pr[0, tk.item()].item())
                cids = torch.cat([cids, tk], dim=-1)
        drafter.deactivate_draft_mode()

        # --- Verify phase: single forward pass of full model on [prefix + draft_tokens] ---
        dt_tensor = torch.tensor([draft_tokens], device=DEVICE)
        vid = torch.cat([iids, dt_tensor], dim=-1)
        with torch.inference_mode():
            fo = model(input_ids=vid, use_cache=False)
            # Get logits at positions corresponding to draft tokens
            # Position iids.shape[1]-1 gives logit for first draft token, etc.
            fl = fo.logits[:, iids.shape[1]-1 : iids.shape[1]-1+k, :].float()

        # --- Accept/reject (Leviathan et al.) ---
        na = 0
        for i in range(k):
            if temperature == 0.0:
                fp = torch.softmax(fl[:, i, :], dim=-1)
                if fl[:, i, :].argmax(dim=-1).item() == draft_tokens[i]:
                    na += 1
                else:
                    break
            else:
                fp = torch.softmax(fl[:, i, :] / temperature, dim=-1)
                pf = fp[0, draft_tokens[i]].item()
                pd_ = draft_probs[i]
                if pd_ == 0: break
                if random.random() < min(1.0, pf / pd_):
                    na += 1
                else:
                    break
        accepted_lengths.append(na)

    ptr = np.array(accepted_lengths) / k if k > 0 else np.zeros(len(accepted_lengths))
    mr, cl, ch = bootstrap_ci(ptr)
    return {"model_key": model_key, "strategy": strategy, "k": k, "temperature": temperature,
            "n_trials": len(accepted_lengths), "acceptance_rate": mr,
            "acceptance_rate_ci_lo": cl, "acceptance_rate_ci_hi": ch,
            "mean_accepted_length": float(np.mean(accepted_lengths)),
            "std_accepted_length": float(np.std(accepted_lengths)),
            "accepted_lengths": accepted_lengths}

In [ ]:
# --- Cell 4B: Run acceptance rate ---
all_acceptance = []
for mk in MODELS_TO_RUN:
    for k in DRAFT_LENGTHS:
        for temp in TEMPERATURES:
            en = f"acceptance_v3__{mk}__k{k}__T{temp}"
            cached = ckpt.load(en)
            if cached is not None and cached.get("acceptance_rate") is not None:
                ckpt.log(f"  {en} from cache"); all_acceptance.append(cached); continue
            try:
                r = simulate_acceptance_rate(mk, k, temperature=temp)
                ckpt.save(en, r); all_acceptance.append(r)
            except Exception as e: ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)

acc_rows = [{"model": r["model_key"], "strategy": r["strategy"], "k": r["k"],
             "temperature": r["temperature"], "acceptance_rate": r["acceptance_rate"],
             "ci_lo": r.get("acceptance_rate_ci_lo", np.nan),
             "ci_hi": r.get("acceptance_rate_ci_hi", np.nan),
             "mean_accepted_length": r["mean_accepted_length"],
             "n_trials": r["n_trials"]}
            for r in all_acceptance if "acceptance_rate" in r]
if acc_rows:
    save_dataframe(pd.DataFrame(acc_rows), "acceptance_rate", index=False)
    display(pd.DataFrame(acc_rows))

In [ ]:
# --- Cell 4C: Figure 2 — Acceptance rate vs k with bootstrap CIs ---
if acc_rows:
    adf = pd.DataFrame(acc_rows)
    t0 = adf[adf["temperature"] == 0.0]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))

    for mk in MODELS_TO_RUN:
        s = t0[t0["model"] == mk].sort_values("k")
        if s.empty: continue
        lb = MODEL_SPECS[mk]["display_name"]
        ln, = a1.plot(s["k"], s["acceptance_rate"], "o-", label=lb)
        c = ln.get_color()
        if "ci_lo" in s.columns:
            a1.fill_between(s["k"], s["ci_lo"], s["ci_hi"], alpha=0.15, color=c)
        a2.plot(s["k"], s["mean_accepted_length"], "s-", label=lb, color=c)

    a1.set_xlabel("Draft length $k$"); a1.set_ylabel("Acceptance rate $\\alpha$")
    a1.set_title("(a) Acceptance rate"); a1.legend(); a1.grid(True, alpha=0.3); a1.set_ylim(0, 1.05)
    a2.set_xlabel("Draft length $k$"); a2.set_ylabel("Mean accepted length $\\bar{\\tau}$")
    a2.set_title("(b) Accepted tokens"); a2.legend(); a2.grid(True, alpha=0.3)
    fig.suptitle("Self-speculative decoding acceptance (T=0) — shaded: 95% bootstrap CI", fontsize=12)
    fig.tight_layout(); save_figure(fig, "paper_acceptance_vs_k")

# Section 5 — Experiment 3: Speedup (theoretical + wall-clock)

**Two separate metrics (required by reviewers):**
1. **Theoretical speedup** = $(\alpha \cdot k + 1) / (1 + k \cdot c_{draft}/c_{full})$ from FLOP ratio
2. **Wall-clock speedup** = tokens/sec (speculative) / tokens/sec (AR baseline with KV cache)

AR baseline uses `model.generate(use_cache=True)` — standard cached inference.
Draft generation attempts `model.generate(use_cache=True)` on patched model; falls back to manual loop.

In [ ]:
# --- Cell 5A: Theoretical speedup ---
def compute_theoretical_speedup(model_key, k, acceptance_rate, seq_len=256):
    """Theoretical speedup from Leviathan et al. (2023).
    
    S_theoretical = (α*k + 1) / (1 + k * c_draft/c_full)
    
    where:
    - α: acceptance rate
    - k: draft length
    - c_draft/c_full: FLOP ratio of draft vs full model
    """
    cr = DraftModelManager.estimate_flop_ratio(model_key, seq_len=seq_len)
    S = (acceptance_rate * k + 1) / (1 + k * cr)
    return S, cr

theo_rows = []
if acc_rows:
    for _, row in pd.DataFrame(acc_rows).iterrows():
        st, cr = compute_theoretical_speedup(row["model"], row["k"], row["acceptance_rate"])
        theo_rows.append({"model": row["model"], "k": row["k"],
                          "temperature": row["temperature"],
                          "acceptance_rate": row["acceptance_rate"],
                          "flop_ratio": cr, "theoretical_speedup": st})
    save_dataframe(pd.DataFrame(theo_rows), "theoretical_speedup", index=False)
    display(pd.DataFrame(theo_rows).query("temperature==0.0"))

In [ ]:
# --- Cell 5B: Wall-clock speedup (v3: KV-cached draft + verify) ---
def _draft_generate_cached(model, tokenizer, inp, k, temperature, strategy_info):
    """Generate k draft tokens using model.generate() with KV cache.
    
    Returns (draft_tokens, draft_probs) or raises exception if cache fails.
    """
    gen_kwargs = dict(
        max_new_tokens=k,
        do_sample=(temperature > 0),
        temperature=max(temperature, 1e-6) if temperature > 0 else None,
        top_k=None, top_p=1.0,
        use_cache=True,
        output_scores=True,
        return_dict_in_generate=True,
    )
    if temperature == 0.0:
        gen_kwargs["do_sample"] = False
        gen_kwargs.pop("temperature", None)

    out = model.generate(**inp, **gen_kwargs)
    inp_len = inp["input_ids"].shape[1]
    draft_tokens = out.sequences[0, inp_len:].tolist()

    # Extract probabilities from scores
    draft_probs = []
    for i, score in enumerate(out.scores):
        if i >= len(draft_tokens): break
        if temperature == 0.0:
            pr = torch.softmax(score.float(), dim=-1)
        else:
            pr = torch.softmax(score.float() / temperature, dim=-1)
        draft_probs.append(pr[0, draft_tokens[i]].item())

    return draft_tokens[:k], draft_probs[:k]


def _draft_generate_manual(model, inp, k, temperature):
    """Fallback: generate k draft tokens without KV cache (manual loop)."""
    iids = inp["input_ids"]
    cids = iids.clone()
    draft_tokens, draft_probs = [], []
    with torch.inference_mode():
        for _ in range(k):
            o = model(input_ids=cids, use_cache=False)
            lg = o.logits[:, -1, :].float()
            pr = torch.softmax(lg, dim=-1) if temperature == 0.0 else torch.softmax(lg / temperature, dim=-1)
            tk = lg.argmax(dim=-1, keepdim=True) if temperature == 0.0 else torch.multinomial(pr, 1)
            draft_tokens.append(tk.item())
            draft_probs.append(pr[0, tk.item()].item())
            cids = torch.cat([cids, tk], dim=-1)
    return draft_tokens, draft_probs


def measure_speedup(model_key, k, temperature=0.0, n_trials=None, max_new_tokens=None):
    """Measure wall-clock speedup of speculative decoding vs AR baseline.
    
    v3 changes:
    - AR baseline: model.generate(use_cache=True) — standard cached inference
    - Draft: model.generate(use_cache=True) on patched model, with fallback to manual
    - Verify: single forward pass per round; KV cache reused across rounds when possible
    - CUDA event timing with warmup
    """
    if n_trials is None: n_trials = SPEEDUP_TRIALS
    if max_new_tokens is None: max_new_tokens = MAX_NEW_TOKENS
    model, tokenizer = load_model_and_tokenizer(model_key)
    arch = MODEL_SPECS[model_key]["arch_family"]
    strategy = ("ssm_only" if arch == "falcon_parallel_hybrid"
                else "linear_only" if arch == "qwen_sequential_hybrid" else "layer_skip")
    mx = get_max_inference_seq_length(model_key)
    mp = min(256, mx - max_new_tokens - 20)

    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    texts = [e["text"].strip() for e in ds if len(e["text"].strip()) > 50][:n_trials]

    drafter = DraftModelManager(model, model_key)

    # Warmup (full model)
    wi = tokenizer(texts[0], return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)
    for _ in range(SPEEDUP_WARMUP_RUNS):
        with torch.inference_mode():
            model.generate(**wi, max_new_tokens=16, do_sample=False, use_cache=True)
    torch.cuda.synchronize()

    # === AR BASELINE (KV-cached, model.generate) ===
    ar_times, ar_tokens = [], []
    for p in tqdm(texts[:n_trials], desc=f"{model_key} AR baseline"):
        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)
        se = torch.cuda.Event(enable_timing=True)
        ee = torch.cuda.Event(enable_timing=True)
        se.record()
        with torch.inference_mode():
            out = model.generate(**inp, max_new_tokens=max_new_tokens,
                                 do_sample=(temperature > 0),
                                 temperature=max(temperature, 1e-6) if temperature > 0 else None,
                                 use_cache=True)
        ee.record(); torch.cuda.synchronize()
        ar_times.append(se.elapsed_time(ee))
        ar_tokens.append(out.shape[1] - inp["input_ids"].shape[1])
    ar_tps = sum(ar_tokens) / (sum(ar_times) / 1000)

    # Determine if cached draft generation works
    draft_cache_works = smoke_results.get(model_key, {}).get("generate_cache_ok", False)
    # Double-check with a quick test
    if not draft_cache_works:
        try:
            drafter.activate_draft_mode(strategy)
            with torch.inference_mode():
                _ = model.generate(**wi, max_new_tokens=4, do_sample=False,
                                    use_cache=True, output_scores=True,
                                    return_dict_in_generate=True)
            draft_cache_works = True
            drafter.deactivate_draft_mode()
        except Exception:
            drafter.deactivate_draft_mode()
            draft_cache_works = False

    # === SPECULATIVE DECODING ===
    sp_times, sp_tokens = [], []
    for p in tqdm(texts[:n_trials], desc=f"{model_key} Spec k={k} ({'cached' if draft_cache_works else 'manual'})"):
        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)
        inp_len = inp["input_ids"].shape[1]
        cids = inp["input_ids"]
        gen = 0

        se = torch.cuda.Event(enable_timing=True)
        ee = torch.cuda.Event(enable_timing=True)
        se.record()

        with torch.inference_mode():
            while gen < max_new_tokens:
                # --- Draft phase ---
                drafter.activate_draft_mode(strategy)
                try:
                    if draft_cache_works:
                        draft_inp = {"input_ids": cids, "attention_mask": torch.ones_like(cids)}
                        draft_tokens, draft_probs = _draft_generate_cached(
                            model, tokenizer, draft_inp, k, temperature, strategy)
                    else:
                        draft_inp = {"input_ids": cids}
                        draft_tokens, draft_probs = _draft_generate_manual(
                            model, draft_inp, k, temperature)
                except Exception:
                    # Ultimate fallback
                    draft_inp = {"input_ids": cids}
                    draft_tokens, draft_probs = _draft_generate_manual(
                        model, draft_inp, k, temperature)
                drafter.deactivate_draft_mode()

                actual_k = len(draft_tokens)
                if actual_k == 0: break

                # --- Verify phase: single forward pass of full model ---
                dt_tensor = torch.tensor([draft_tokens], device=DEVICE)
                vi = torch.cat([cids, dt_tensor], dim=-1)
                vo = model(input_ids=vi, use_cache=False)
                # Logits at positions [prefix_len-1 : prefix_len-1+k] correspond to draft tokens
                fl = vo.logits[:, cids.shape[1]-1 : cids.shape[1]-1+actual_k, :].float()

                # --- Accept/reject (Leviathan et al.) ---
                na = 0
                for i in range(actual_k):
                    if temperature == 0.0:
                        if fl[:, i, :].argmax(dim=-1).item() == draft_tokens[i]:
                            na += 1
                        else:
                            break
                    else:
                        fp = torch.softmax(fl[:, i, :] / temperature, dim=-1)
                        pf = fp[0, draft_tokens[i]].item()
                        pd_ = draft_probs[i]
                        if pd_ > 0 and random.random() < min(1.0, pf / pd_):
                            na += 1
                        else:
                            break

                # Collect accepted tokens + correction/bonus token
                if na < actual_k:
                    # Rejection: sample correction token from verify logits at rejection position
                    bl = fl[:, na, :]
                    if temperature == 0.0:
                        bonus = bl.argmax(dim=-1, keepdim=True)
                    else:
                        bonus = torch.multinomial(torch.softmax(bl / temperature, dim=-1), 1)
                    acc = draft_tokens[:na] + [bonus.item()]
                else:
                    # All accepted: sample bonus token from next position
                    ll = vo.logits[:, -1, :].float()
                    if temperature == 0.0:
                        bonus = ll.argmax(dim=-1, keepdim=True)
                    else:
                        bonus = torch.multinomial(torch.softmax(ll / temperature, dim=-1), 1)
                    acc = draft_tokens + [bonus.item()]

                cids = torch.cat([cids, torch.tensor([acc], device=DEVICE)], dim=-1)
                gen += len(acc)

        ee.record(); torch.cuda.synchronize()
        sp_times.append(se.elapsed_time(ee))
        sp_tokens.append(gen)

    sp_tps = sum(sp_tokens) / (sum(sp_times) / 1000)

    return {"model_key": model_key, "strategy": strategy, "k": k, "temperature": temperature,
            "ar_tok_per_sec": ar_tps, "spec_tok_per_sec": sp_tps,
            "speedup": sp_tps / ar_tps if ar_tps > 0 else 0,
            "n_trials": n_trials, "draft_cache_used": draft_cache_works,
            "ar_times_ms": ar_times, "sp_times_ms": sp_times}

In [ ]:
# --- Cell 5C: Run speedup ---
speedup_results = []
SPEEDUP_K_VALUES = [2, 4, 8] if PAPER_MODE else [4]
for mk in MODELS_TO_RUN:
    for k in SPEEDUP_K_VALUES:
        en = f"speedup_v3__{mk}__k{k}"
        cached = ckpt.load(en)
        if cached is not None and cached.get("speedup") is not None:
            ckpt.log(f"  {en} from cache"); speedup_results.append(cached); continue
        try:
            r = measure_speedup(mk, k)
            ckpt.save(en, r); speedup_results.append(r)
            ckpt.log(f"  {mk} k={k}: {r['speedup']:.3f}x (draft_cache={r['draft_cache_used']})")
        except Exception as e: ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)

sp_rows = [{"model": r["model_key"], "strategy": r["strategy"], "k": r["k"],
            "ar_tok_per_sec": r["ar_tok_per_sec"], "spec_tok_per_sec": r["spec_tok_per_sec"],
            "speedup": r["speedup"], "draft_cache": r.get("draft_cache_used", False)}
           for r in speedup_results if "speedup" in r]
if sp_rows:
    save_dataframe(pd.DataFrame(sp_rows), "speedup", index=False)
    display(pd.DataFrame(sp_rows))

In [ ]:
# --- Cell 5D: Figure 3 — Theoretical vs empirical speedup (side by side) ---
if theo_rows and sp_rows:
    tdf = pd.DataFrame(theo_rows); sdf = pd.DataFrame(sp_rows)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))

    t0t = tdf[tdf["temperature"] == 0.0]
    for mk in MODELS_TO_RUN:
        s = t0t[t0t["model"] == mk].sort_values("k")
        if not s.empty:
            a1.plot(s["k"], s["theoretical_speedup"], "o-", label=MODEL_SPECS[mk]["display_name"])
    a1.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
    a1.set_xlabel("Draft length $k$"); a1.set_ylabel("Theoretical speedup")
    a1.set_title("(a) Theoretical (FLOP-based)"); a1.legend(); a1.grid(True, alpha=0.3)

    mlist = sorted(sdf["model"].unique()); ks = sorted(sdf["k"].unique())
    w = 0.8 / max(len(mlist), 1)
    for i, mk in enumerate(mlist):
        s = sdf[sdf["model"] == mk].set_index("k").reindex(ks)
        bars = a2.bar(np.arange(len(ks)) + i*w, s["speedup"], width=w,
                      label=MODEL_SPECS.get(mk, {}).get("display_name", mk), alpha=0.8)
        # Overlay theoretical as black stars
        for j, kv in enumerate(ks):
            tv = t0t[(t0t["model"] == mk) & (t0t["k"] == kv)]["theoretical_speedup"]
            if not tv.empty:
                a2.plot(j + i*w, tv.values[0], "k*", markersize=10, zorder=5)
    a2.set_xticks(np.arange(len(ks)) + w*(len(mlist)-1)/2)
    a2.set_xticklabels([f"k={k}" for k in ks])
    a2.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
    a2.set_ylabel("Speedup"); a2.set_title("(b) Wall-clock (bars) vs theoretical (★)")
    a2.legend(); a2.grid(True, alpha=0.3, axis="y")
    fig.suptitle("Theoretical vs. empirical speedup (T=0)", fontsize=13)
    fig.tight_layout(); save_figure(fig, "paper_speedup_theo_vs_empirical")

    # Additional: speedup gap analysis
    print("\n--- Theoretical vs Empirical Gap ---")
    for mk in MODELS_TO_RUN:
        t_vals = t0t[t0t["model"] == mk].set_index("k")["theoretical_speedup"]
        e_vals = sdf[sdf["model"] == mk].set_index("k")["speedup"]
        for kv in ks:
            if kv in t_vals.index and kv in e_vals.index:
                t, e = t_vals[kv], e_vals[kv]
                gap = (t - e) / t * 100
                print(f"  {mk} k={kv}: theo={t:.3f}x emp={e:.3f}x gap={gap:.1f}%")

# Section 6 — Experiment 4: Task-dependent acceptance rate
Tests acceptance rate across different task domains (MMLU, GSM8K, Alpaca).

Produces: `task_acceptance.csv`, Figure 4 (heatmap).

In [ ]:
# --- Cell 6A: Task-dependent analysis ---
def get_task_prompts(tk, n):
    src = TASK_PROMPTS[tk]["prompts_source"]
    if src == "mmlu":
        ds = load_dataset("cais/mmlu", "all", split="test")
        ps = [f"Question: {e['question']}\nAnswer:" for e in ds]
    elif src == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main", split="test")
        ps = [f"Question: {e['question']}\nAnswer:" for e in ds]
    elif src == "alpaca":
        ds = load_dataset("tatsu-lab/alpaca", split="train")
        ps = [e["instruction"] for e in ds if len(e["instruction"]) > 20]
    else:
        ps = [f"Tell me about topic {i}" for i in range(n)]
    random.seed(SEED); random.shuffle(ps)
    return ps[:n]

task_acceptance_results = []
k_for_tasks = 4
for mk in MODELS_TO_RUN:
    for tk in TASK_PROMPTS:
        en = f"task_accept_v3__{mk}__{tk}"
        cached = ckpt.load(en)
        if cached is not None and cached.get("acceptance_rate") is not None:
            cached["task"] = tk; task_acceptance_results.append(cached); continue
        try:
            ps = get_task_prompts(tk, TASK_PROMPTS[tk]["n_prompts"])
            r = simulate_acceptance_rate(mk, k_for_tasks, temperature=0.0,
                                          n_trials=len(ps), prompts=ps)
            r["task"] = tk; ckpt.save(en, r); task_acceptance_results.append(r)
        except Exception as e: ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)

task_rows = [{"model": r["model_key"], "task": r["task"],
              "acceptance_rate": r["acceptance_rate"],
              "ci_lo": r.get("acceptance_rate_ci_lo", np.nan),
              "ci_hi": r.get("acceptance_rate_ci_hi", np.nan),
              "mean_accepted_length": r["mean_accepted_length"]}
             for r in task_acceptance_results if "acceptance_rate" in r]
if task_rows:
    tdf = pd.DataFrame(task_rows)
    save_dataframe(tdf, "task_acceptance", index=False)
    display(tdf)

    # Heatmap
    pv = tdf.pivot(index="model", columns="task", values="acceptance_rate")
    fig, ax = plt.subplots(figsize=(8, max(4, len(pv.index) * 1.2)))
    im = ax.imshow(pv.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pv.columns))); ax.set_xticklabels(pv.columns)
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels([MODEL_SPECS.get(m, {}).get("display_name", m) for m in pv.index])
    for i in range(len(pv.index)):
        for j in range(len(pv.columns)):
            v = pv.values[i, j]
            ax.text(j, i, f"{v:.2f}" if not np.isnan(v) else "--",
                    ha="center", va="center", fontsize=11, fontweight="bold",
                    color="white" if v < 0.4 else "black")
    fig.colorbar(im, ax=ax, label="Acceptance rate (k=4)")
    ax.set_title("Task-dependent acceptance rate (greedy, k=4)")
    fig.tight_layout(); save_figure(fig, "paper_task_acceptance_heatmap")

# Section 7 — Experiment 5: Optimal $k^*$, Paper 2 correlation, and output quality

Produces: `optimal_k.csv`, `paper2_correlation.csv`, `quality_check.csv`, Figure 5.

In [ ]:
# --- Cell 7A: Optimal k* + Paper 2 correlation ---
if theo_rows:
    tdf = pd.DataFrame(theo_rows)
    t0t = tdf[tdf["temperature"] == 0.0]
    orows = []
    for mk in MODELS_TO_RUN:
        s = t0t[t0t["model"] == mk]
        if s.empty: continue
        b = s.loc[s["theoretical_speedup"].idxmax()]
        es = None
        if sp_rows:
            em = [r for r in sp_rows if r["model"] == mk and r["k"] == int(b["k"])]
            if em: es = em[0]["speedup"]
        orows.append({"model": mk, "display_name": MODEL_SPECS[mk]["display_name"],
                       "optimal_k": int(b["k"]),
                       "theo_speedup": b["theoretical_speedup"],
                       "emp_speedup": es,
                       "acceptance": b["acceptance_rate"],
                       "flop_ratio": b["flop_ratio"]})
    if orows:
        odf = pd.DataFrame(orows)
        save_dataframe(odf, "optimal_k", index=False)
        display(odf)

# Paper 2 correlation: PPL degradation from ablation predicts draft quality
p2_data = {
    "qwen3.5-0.8b": {"bl_ppl": 7.623609917712736, "no_attn_ppl": 624.8427387029609},
    "falcon-h1-0.5b": {"bl_ppl": 5.621297448593752, "no_attn_ppl": 17.725424121461643},
}
corr_rows = []
if acc_rows:
    adf = pd.DataFrame(acc_rows)
    t0 = adf[adf["temperature"] == 0.0]
    for mk in ["qwen3.5-0.8b", "falcon-h1-0.5b"]:
        info = p2_data.get(mk, {})
        ratio = info.get("no_attn_ppl", 0) / info.get("bl_ppl", 1)
        a = t0[(t0["model"] == mk) & (t0["k"] == 4)]["acceptance_rate"].values
        if len(a) > 0:
            corr_rows.append({"model": mk,
                              "display_name": MODEL_SPECS[mk]["display_name"],
                              "ppl_baseline": info["bl_ppl"],
                              "ppl_no_attn": info["no_attn_ppl"],
                              "ppl_ratio": ratio,
                              "log10_ppl_ratio": np.log10(ratio),
                              "alpha_k4": a[0]})
if corr_rows:
    cdf = pd.DataFrame(corr_rows)
    save_dataframe(cdf, "paper2_correlation", index=False)
    display(cdf)
    print(f"\nPearson r = {sp_stats.pearsonr(cdf['log10_ppl_ratio'], cdf['alpha_k4'])}" if len(cdf) > 1 else "")

    # Figure 5: Scatter plot
    fig, ax = plt.subplots(figsize=(7, 5))
    for _, r in cdf.iterrows():
        lb = r["display_name"]
        ax.scatter(r["log10_ppl_ratio"], r["alpha_k4"], s=150, zorder=5, label=lb)
        ax.annotate(f'  {lb}\n  PPL ×{r["ppl_ratio"]:.1f}',
                    (r["log10_ppl_ratio"], r["alpha_k4"]), fontsize=9)
    ax.set_xlabel("$\\log_{10}$(PPL degradation without attention)")
    ax.set_ylabel("Acceptance rate $\\alpha$ (k=4)")
    ax.set_title("Paper 2 → Paper 4: Ablation predicts draft quality")
    ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()
    save_figure(fig, "paper_ppl_vs_acceptance")

In [ ]:
# --- Cell 7B: Output quality verification (lossless check) ---
def verify_output_quality(model_key, n_prompts=50, max_new_tokens=32):
    """Verify that speculative decoding produces identical output to AR (greedy).
    
    This is the LOSSLESS guarantee: with T=0 (greedy), speculative decoding
    must produce exactly the same tokens as standard AR decoding.
    """
    model, tokenizer = load_model_and_tokenizer(model_key)
    arch = MODEL_SPECS[model_key]["arch_family"]
    strategy = ("ssm_only" if arch == "falcon_parallel_hybrid"
                else "linear_only" if arch == "qwen_sequential_hybrid" else "layer_skip")
    mx = get_max_inference_seq_length(model_key)

    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    texts = [e["text"].strip() for e in ds if len(e["text"].strip()) > 100][:n_prompts]

    drafter = DraftModelManager(model, model_key)
    matches = 0; total = 0; mismatches = []

    for text in tqdm(texts, desc=f"{model_key} quality check"):
        inp = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=min(256, mx - max_new_tokens - 10)).to(DEVICE)
        inp_len = inp["input_ids"].shape[1]

        # AR baseline (greedy)
        with torch.inference_mode():
            ar_out = model.generate(**inp, max_new_tokens=max_new_tokens,
                                     do_sample=False, use_cache=True)
        ar_tokens = ar_out[0, inp_len:].tolist()

        # Speculative decoding (greedy)
        cids = inp["input_ids"]; gen_tokens = []
        with torch.inference_mode():
            while len(gen_tokens) < max_new_tokens:
                k = 4
                drafter.activate_draft_mode(strategy)
                dt, dp = [], []; did = cids.clone()
                for _ in range(k):
                    o = model(input_ids=did, use_cache=False)
                    lg = o.logits[:, -1, :].float()
                    pr = torch.softmax(lg, dim=-1); tk = lg.argmax(dim=-1, keepdim=True)
                    dt.append(tk.item()); dp.append(pr[0, tk.item()].item())
                    did = torch.cat([did, tk], dim=-1)
                drafter.deactivate_draft_mode()

                dtt = torch.tensor([dt], device=DEVICE)
                vi = torch.cat([cids, dtt], dim=-1)
                fo = model(input_ids=vi, use_cache=False)
                fl = fo.logits[:, cids.shape[1]-1:cids.shape[1]-1+k, :].float()

                na = 0
                for i in range(k):
                    if fl[:, i, :].argmax(dim=-1).item() == dt[i]: na += 1
                    else: break
                if na < k:
                    bonus = fl[:, na, :].argmax(dim=-1).item()
                    acc = dt[:na] + [bonus]
                else:
                    ll = fo.logits[:, -1, :].float()
                    bonus = ll.argmax(dim=-1).item()
                    acc = dt + [bonus]
                gen_tokens.extend(acc)
                cids = torch.cat([cids, torch.tensor([acc], device=DEVICE)], dim=-1)

        spec_tokens = gen_tokens[:max_new_tokens]

        # Compare
        match_len = min(len(ar_tokens), len(spec_tokens))
        is_match = ar_tokens[:match_len] == spec_tokens[:match_len]
        if is_match: matches += 1
        else:
            # Find first mismatch position
            for j in range(match_len):
                if ar_tokens[j] != spec_tokens[j]:
                    mismatches.append({"pos": j, "ar": ar_tokens[j], "spec": spec_tokens[j]})
                    break
        total += 1

    match_rate = matches / total if total > 0 else 0
    return {"model_key": model_key, "strategy": strategy,
            "match_rate": match_rate, "matches": matches, "total": total,
            "n_mismatches": len(mismatches), "first_mismatches": mismatches[:5]}

quality_results = []
for mk in MODELS_TO_RUN:
    en = f"quality_v3__{mk}"
    cached = ckpt.load(en)
    if cached is not None and "match_rate" in cached:
        ckpt.log(f"  {en} from cache"); quality_results.append(cached); continue
    try:
        r = verify_output_quality(mk)
        ckpt.save(en, r); quality_results.append(r)
        ckpt.log(f"  {mk}: {r['match_rate']:.1%} output match ({r['matches']}/{r['total']})")
    except Exception as e: ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)

if quality_results:
    qdf = pd.DataFrame([{"model": r["model_key"], "strategy": r["strategy"],
                          "match_rate": r["match_rate"], "mismatches": r["n_mismatches"]}
                         for r in quality_results])
    save_dataframe(qdf, "quality_check", index=False)
    display(qdf)
    all_match = all(r["match_rate"] == 1.0 for r in quality_results)
    print(f"\nLossless guarantee: {'VERIFIED ✓' if all_match else 'PARTIAL — see mismatches above'}")

# Section 8 — Experiment 6: Additional draft strategies (early-exit on hybrids)

Tests early-exit on hybrid models as comparison baseline. This answers: is component-aware self-speculation better than generic strategies?

In [ ]:
# --- Cell 8A: Early-exit + LayerSkip comparison on hybrid models ---
comparison_results = []
strategies_to_compare = ["early_exit", "layer_skip"]

for mk in [m for m in MODELS_TO_RUN if MODEL_SPECS[m]["hybrid_type"] != "transformer"]:
    for strat in strategies_to_compare:
        en = f"comparison_v3__{mk}__{strat}"
        cached = ckpt.load(en)
        if cached is not None and "acceptance_rate" in cached:
            ckpt.log(f"  {en} from cache"); comparison_results.append(cached); continue
        try:
            model, tokenizer = load_model_and_tokenizer(mk)
            drafter = DraftModelManager(model, mk)
            # Temporarily override strategy detection
            ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")
            prompts = [e["text"].strip() for e in ds if len(e["text"].strip())>100][:100]
            mx = get_max_inference_seq_length(mk)

            accepted = []
            k = 4
            for text in tqdm(prompts, desc=f"{mk} {strat}"):
                inp = tokenizer(text, return_tensors="pt", truncation=True,
                                max_length=min(512, mx-k-10)).to(DEVICE)
                iids = inp["input_ids"]

                drafter.activate_draft_mode(strat)
                dt, dp = [], []; cids = iids.clone()
                with torch.inference_mode():
                    for _ in range(k):
                        o = model(input_ids=cids, use_cache=False)
                        lg = o.logits[:,-1,:].float()
                        pr = torch.softmax(lg, dim=-1); tk = lg.argmax(dim=-1, keepdim=True)
                        dt.append(tk.item()); dp.append(pr[0,tk.item()].item())
                        cids = torch.cat([cids, tk], dim=-1)
                drafter.deactivate_draft_mode()

                vid = torch.cat([iids, torch.tensor([dt], device=DEVICE)], dim=-1)
                with torch.inference_mode():
                    fo = model(input_ids=vid, use_cache=False)
                    fl = fo.logits[:, iids.shape[1]-1:iids.shape[1]-1+k, :].float()

                na = 0
                for i in range(k):
                    if fl[:,i,:].argmax(dim=-1).item() == dt[i]: na += 1
                    else: break
                accepted.append(na)

            arr = np.array(accepted) / k
            mr, cl, ch = bootstrap_ci(arr)
            r = {"model_key": mk, "strategy": strat, "k": k, "acceptance_rate": mr,
                 "ci_lo": cl, "ci_hi": ch, "mean_accepted_length": float(np.mean(accepted)),
                 "n_trials": len(accepted)}
            ckpt.save(en, r); comparison_results.append(r)
        except Exception as e:
            ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)

if comparison_results:
    # Combine with main results for comparison plot
    comp_df = pd.DataFrame(comparison_results)
    save_dataframe(comp_df, "strategy_comparison", index=False)
    display(comp_df)

    # Comparison bar chart: component-aware vs early-exit vs layer-skip
    fig, ax = plt.subplots(figsize=(10, 5))
    hybrid_models = [m for m in MODELS_TO_RUN if MODEL_SPECS[m]["hybrid_type"] != "transformer"]
    x = np.arange(len(hybrid_models)); w = 0.25

    # Component-aware (from main acceptance results)
    ca_rates = []
    for mk in hybrid_models:
        if acc_rows:
            am = [r for r in acc_rows if r["model"]==mk and r["k"]==4 and r["temperature"]==0.0]
            ca_rates.append(am[0]["acceptance_rate"] if am else 0)
        else:
            ca_rates.append(0)
    ax.bar(x - w, ca_rates, w, label="Component-aware", alpha=0.8)

    for i, strat in enumerate(strategies_to_compare):
        rates = []
        for mk in hybrid_models:
            cr = [r for r in comparison_results if r["model_key"]==mk and r["strategy"]==strat]
            rates.append(cr[0]["acceptance_rate"] if cr else 0)
        ax.bar(x + i*w, rates, w, label=strat.replace("_", " ").title(), alpha=0.8)

    ax.set_xticks(x); ax.set_xticklabels([MODEL_SPECS[m]["display_name"] for m in hybrid_models])
    ax.set_ylabel("Acceptance rate (k=4, T=0)"); ax.set_ylim(0, 1.05)
    ax.set_title("Component-aware vs generic self-speculation strategies")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")
    fig.tight_layout(); save_figure(fig, "paper_strategy_comparison")

# Section 9 — Summary and export

In [ ]:
# --- Cell 9A: Summary table (main results) ---
print("\n" + "="*80 + "\nPAPER 4 — RESULTS SUMMARY (v3)\n" + "="*80)
rows = []
for mk in MODELS_TO_RUN:
    sp = MODEL_SPECS[mk]
    dv = divergence_results.get(mk, {})

    # Best theoretical speedup
    bt = None
    if theo_rows:
        mk_theo = [r for r in theo_rows if r["model"]==mk and r["temperature"]==0.0]
        if mk_theo: bt = max(mk_theo, key=lambda r: r["theoretical_speedup"])

    # Best empirical speedup
    be = None
    if sp_rows:
        mk_emp = [r for r in sp_rows if r["model"]==mk]
        if mk_emp: be = max(mk_emp, key=lambda r: r["speedup"])

    # Acceptance rate at k=4
    a4, ci = None, ""
    if acc_rows:
        am = [r for r in acc_rows if r["model"]==mk and r["k"]==4 and r["temperature"]==0.0]
        if am:
            a4 = am[0]["acceptance_rate"]
            cl, ch = am[0].get("ci_lo", np.nan), am[0].get("ci_hi", np.nan)
            if not np.isnan(cl): ci = f" [{cl:.3f},{ch:.3f}]"

    fr = DraftModelManager.estimate_flop_ratio(mk)
    draft_gf, full_gf = DraftModelManager.estimate_flops_gflops(mk)

    draft_strat = ("SSM-only" if sp["hybrid_type"]=="parallel"
                   else "Linear-only" if sp["hybrid_type"]=="sequential"
                   else "LayerSkip-33%")

    rows.append({
        "Model": sp["display_name"],
        "Arch": sp["hybrid_type"],
        "Draft": draft_strat,
        "FLOP_r": f"{fr:.3f}",
        "GFLOPs(d/f)": f"{draft_gf:.1f}/{full_gf:.1f}",
        "D_TV": f"{dv.get('d_tv_mean', float('nan')):.3f}",
        "α(k=4)": f"{a4:.3f}{ci}" if a4 else "--",
        "S_theo": f"{bt['theoretical_speedup']:.2f}x" if bt else "--",
        "S_emp": f"{be['speedup']:.2f}x" if be else "--",
        "k*": int(bt["k"]) if bt else "--",
    })

sdf = pd.DataFrame(rows)
display(sdf)
lx = sdf.to_latex(index=False, escape=False)
print("\n" + lx)
save_latex_table(lx, "paper_table1_main_results")

In [ ]:
# --- Cell 9B: Export manifest + sanity checks ---
import glob
print("\n" + "="*80 + "\nEXPORT MANIFEST\n" + "="*80)
for dn, lb in [(RESULTS_DIR, "CSVs"), (FIGURES_DIR, "Figures"), (TABLES_DIR, "Tables")]:
    fs = sorted(glob.glob(os.path.join(dn, "*")))
    print(f"\n{lb} ({len(fs)}):")
    for f in fs: print(f"  {os.path.basename(f):55s} {os.path.getsize(f):>10,} bytes")
print(f"\nCheckpoints: {len(ckpt.list_checkpoints())}")

print("\n" + "="*80 + "\nSANITY CHECKS\n" + "="*80)
checks = {
    "Divergence": len(divergence_results) >= len(MODELS_CORE),
    "Acceptance": bool(acc_rows),
    "Theo speedup": bool(theo_rows),
    "Emp speedup": bool(sp_rows),
    "Task-dep": bool(task_rows),
    "Paper2 corr": bool(corr_rows),
    "Bootstrap CIs": any(not np.isnan(r.get("ci_lo", np.nan)) for r in acc_rows) if acc_rows else False,
    "Quality check": bool(quality_results),
    "Strategy comparison": bool(comparison_results),
    "Draft cache": any(r.get("draft_cache", False) for r in sp_rows) if sp_rows else "N/A",
}
for n, p in checks.items():
    status = "PASS" if p else ("N/A" if p == "N/A" else "FAIL")
    print(f"  [{status}] {n}")

print("\nAll experiments complete. Ready for paper writing.")
print(f"Total checkpoints: {len(ckpt.list_checkpoints())}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
# --- Speedup measurement v2 (corrected draft KV cache + verify indexing) ---# This is the corrected version used for final paper results.# ==================================================================# CELL A: Speedup v2 (fixes draft KV cache + verify indexing)# ================================================================== def measure_speedup_v2(model_key, k, temperature=0.0, n_trials=None, max_new_tokens=None):    if n_trials is None: n_trials = SPEEDUP_TRIALS    if max_new_tokens is None: max_new_tokens = MAX_NEW_TOKENS    model, tokenizer = load_model_and_tokenizer(model_key)    arch = MODEL_SPECS[model_key]["arch_family"]    strategy = ("ssm_only" if arch == "falcon_parallel_hybrid"                else "linear_only" if arch == "qwen_sequential_hybrid" else "layer_skip")    mx = get_max_inference_seq_length(model_key)    mp = min(256, mx - max_new_tokens - 20)    ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")    texts = [e["text"].strip() for e in ds if len(e["text"].strip()) > 50][:n_trials]    drafter = DraftModelManager(model, model_key)     # Warmup    wi = tokenizer(texts[0], return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)    for _ in range(SPEEDUP_WARMUP_RUNS):        with torch.inference_mode():            try:                model.generate(**wi, max_new_tokens=16, do_sample=False, use_cache=True)            except Exception:                _ = model(**wi, use_cache=False)    torch.cuda.synchronize()     # AR BASELINE (KV-cached)    ar_times, ar_tokens = [], []    for p in tqdm(texts[:n_trials], desc=f"{model_key} AR baseline"):        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)        se = torch.cuda.Event(enable_timing=True); ee = torch.cuda.Event(enable_timing=True)        se.record()        with torch.inference_mode():            out = model.generate(**inp, max_new_tokens=max_new_tokens,                                 do_sample=(temperature > 0),                                 temperature=max(temperature, 1e-6) if temperature > 0 else None,                                 use_cache=True)        ee.record(); torch.cuda.synchronize()        ar_times.append(se.elapsed_time(ee))        ar_tokens.append(out.shape[1] - inp["input_ids"].shape[1])    ar_tps = sum(ar_tokens) / (sum(ar_times) / 1000)     # SPECULATIVE DECODING (draft with incremental KV cache)    sp_times, sp_tokens = [], []    for p in tqdm(texts[:n_trials], desc=f"{model_key} Spec k={k}"):        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=mp).to(DEVICE)        cids = inp["input_ids"]; gen = 0        se = torch.cuda.Event(enable_timing=True); ee = torch.cuda.Event(enable_timing=True)        se.record()        with torch.inference_mode():            while gen < max_new_tokens:                # Draft phase with KV cache attempt                drafter.activate_draft_mode(strategy)                draft_tokens, draft_probs = [], []                draft_past, draft_input = None, cids                for step in range(k):                    try:                        out = model(input_ids=draft_input, past_key_values=draft_past, use_cache=True)                        draft_past = out.past_key_values                    except Exception:                        full_seq = torch.cat([cids] + [torch.tensor([[t]], device=DEVICE) for t in draft_tokens], dim=-1) if draft_tokens else cids                        out = model(input_ids=full_seq, use_cache=False)                        draft_past = None                    lg = out.logits[:, -1, :].float()                    pr = torch.softmax(lg, dim=-1)                    if temperature == 0.0:                        tk = lg.argmax(dim=-1, keepdim=True)                    else:                        tk = torch.multinomial(torch.softmax(lg / temperature, dim=-1), 1)                    draft_tokens.append(tk.item())                    draft_probs.append(pr[0, tk.item()].item())                    draft_input = tk                del draft_past                drafter.deactivate_draft_mode()                 actual_k = len(draft_tokens)                if actual_k == 0: break                 # Verify phase                dt_tensor = torch.tensor([draft_tokens], device=DEVICE)                vi = torch.cat([cids, dt_tensor], dim=-1)                vo = model(input_ids=vi, use_cache=False)                fl = vo.logits[:, cids.shape[1]-1 : cids.shape[1]-1+actual_k, :].float()                 # Accept/reject                na = 0                for i in range(actual_k):                    if temperature == 0.0:                        if fl[:, i, :].argmax(dim=-1).item() == draft_tokens[i]:                            na += 1                        else: break                    else:                        fp = torch.softmax(fl[:, i, :] / temperature, dim=-1)                        pf = fp[0, draft_tokens[i]].item()                        pd_ = draft_probs[i]                        if pd_ > 0 and random.random() < min(1.0, pf / pd_):                            na += 1                        else: break                 if na < actual_k:                    bonus = fl[:, na, :].argmax(dim=-1, keepdim=True) if temperature == 0.0 else torch.multinomial(torch.softmax(fl[:, na, :] / temperature, dim=-1), 1)                    acc = draft_tokens[:na] + [bonus.item()]                else:                    ll = vo.logits[:, -1, :].float()                    bonus = ll.argmax(dim=-1, keepdim=True) if temperature == 0.0 else torch.multinomial(torch.softmax(ll / temperature, dim=-1), 1)                    acc = draft_tokens + [bonus.item()]                 cids = torch.cat([cids, torch.tensor([acc], device=DEVICE)], dim=-1)                gen += len(acc)        ee.record(); torch.cuda.synchronize()        sp_times.append(se.elapsed_time(ee)); sp_tokens.append(gen)     sp_tps = sum(sp_tokens) / (sum(sp_times) / 1000)    return {"model_key": model_key, "strategy": strategy, "k": k, "temperature": temperature,            "ar_tok_per_sec": ar_tps, "spec_tok_per_sec": sp_tps,            "speedup": sp_tps / ar_tps if ar_tps > 0 else 0, "n_trials": n_trials} # Run# Runspeedup_results_v2 = []for mk in MODELS_TO_RUN:    trials = 10 if "qwen3.5" in mk else SPEEDUP_TRIALS    for k in [2, 4, 8]:        en = f"speedup_v3fix__{mk}__k{k}"        cached = ckpt.load(en)        if cached is not None and cached.get("speedup") is not None:            ckpt.log(f"  {en} from cache"); speedup_results_v2.append(cached); continue        try:            r = measure_speedup_v2(mk, k, n_trials=trials)            ckpt.save(en, r); speedup_results_v2.append(r)            ckpt.log(f"  {mk} k={k}: {r['speedup']:.3f}x")        except Exception as e:            ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()    unload_model(mk)sp_rows_v2 = [{"model": r["model_key"], "strategy": r["strategy"], "k": r["k"],               "ar_tok_per_sec": r["ar_tok_per_sec"], "spec_tok_per_sec": r["spec_tok_per_sec"],               "speedup": r["speedup"]} for r in speedup_results_v2 if "speedup" in r]if sp_rows_v2:    save_dataframe(pd.DataFrame(sp_rows_v2), "speedup_v2", index=False)    display(pd.DataFrame(sp_rows_v2)) 

In [ ]:
# ==================================================================
# CELL B: Quality check v2 (consistent conditions: both without cache)
# ==================================================================
 
def verify_output_quality_v2(model_key, n_prompts=50, max_new_tokens=32):
    model, tokenizer = load_model_and_tokenizer(model_key)
    arch = MODEL_SPECS[model_key]["arch_family"]
    strategy = ("ssm_only" if arch == "falcon_parallel_hybrid"
                else "linear_only" if arch == "qwen_sequential_hybrid" else "layer_skip")
    mx = get_max_inference_seq_length(model_key)
    drafter = DraftModelManager(model, model_key)
    ds = load_dataset("wikitext","wikitext-2-raw-v1",split="validation")
    texts = [e["text"].strip() for e in ds if len(e["text"].strip()) > 100][:n_prompts]
    matches = 0; total = 0; mismatches = []
 
    for text in tqdm(texts, desc=f"{model_key} quality v2"):
        inp = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=min(256, mx - max_new_tokens - 10)).to(DEVICE)
        iids = inp["input_ids"]
 
        # AR: manual greedy, NO cache (same conditions as spec)
        ar_tokens = []; ar_ids = iids.clone()
        with torch.inference_mode():
            for _ in range(max_new_tokens):
                out = model(input_ids=ar_ids, use_cache=False)
                next_tok = out.logits[:, -1, :].float().argmax(dim=-1).item()
                ar_tokens.append(next_tok)
                ar_ids = torch.cat([ar_ids, torch.tensor([[next_tok]], device=DEVICE)], dim=-1)
 
        # Speculative: greedy
        spec_tokens = []; cids = iids.clone(); k = 4
        with torch.inference_mode():
            while len(spec_tokens) < max_new_tokens:
                drafter.activate_draft_mode(strategy)
                dt = []; did = cids.clone()
                for _ in range(k):
                    o = model(input_ids=did, use_cache=False)
                    tk = o.logits[:, -1, :].float().argmax(dim=-1).item()
                    dt.append(tk); did = torch.cat([did, torch.tensor([[tk]], device=DEVICE)], dim=-1)
                drafter.deactivate_draft_mode()
 
                dtt = torch.tensor([dt], device=DEVICE)
                vi = torch.cat([cids, dtt], dim=-1)
                fo = model(input_ids=vi, use_cache=False)
                fl = fo.logits[:, cids.shape[1]-1 : cids.shape[1]-1+k, :].float()
 
                na = 0
                for i in range(k):
                    if fl[:, i, :].argmax(dim=-1).item() == dt[i]: na += 1
                    else: break
 
                if na < k:
                    correction = fl[:, na, :].argmax(dim=-1).item()
                    acc = dt[:na] + [correction]
                else:
                    bonus = fo.logits[:, -1, :].float().argmax(dim=-1).item()
                    acc = dt + [bonus]
                spec_tokens.extend(acc)
                cids = torch.cat([cids, torch.tensor([acc], device=DEVICE)], dim=-1)
 
        spec_tokens = spec_tokens[:max_new_tokens]
        match_len = min(len(ar_tokens), len(spec_tokens))
        is_match = ar_tokens[:match_len] == spec_tokens[:match_len]
        if is_match: matches += 1
        else:
            for j in range(match_len):
                if ar_tokens[j] != spec_tokens[j]:
                    mismatches.append({"pos": j, "ar": tokenizer.decode([ar_tokens[j]]),
                                       "spec": tokenizer.decode([spec_tokens[j]])})
                    break
        total += 1
 
    rate = matches / total if total > 0 else 0
    print(f"\n{'='*60}")
    print(f"QUALITY: {model_key} | {rate:.1%} match ({matches}/{total})")
    if mismatches: print(f"  First mismatch: pos={mismatches[0]['pos']}, AR='{mismatches[0]['ar']}' vs Spec='{mismatches[0]['spec']}'")
    print(f"{'='*60}")
    return {"model_key": model_key, "strategy": strategy, "match_rate": rate,
            "matches": matches, "total": total, "n_mismatches": len(mismatches)}
 
# Run
quality_results_v2 = []
for mk in MODELS_TO_RUN:
    en = f"quality_v3fix__{mk}"
    cached = ckpt.load(en)
    if cached is not None and "match_rate" in cached:
        ckpt.log(f"  {en} from cache"); quality_results_v2.append(cached); continue
    try:
        r = verify_output_quality_v2(mk)
        ckpt.save(en, r); quality_results_v2.append(r)
    except Exception as e: ckpt.log(f"[ERROR] {en}: {e}"); traceback.print_exc()
    unload_model(mk)
 
if quality_results_v2:
    qdf = pd.DataFrame([{"model": r["model_key"], "strategy": r["strategy"],
                          "match_rate": r["match_rate"], "mismatches": r["n_mismatches"]}
                         for r in quality_results_v2])
    save_dataframe(qdf, "quality_check_v2", index=False)
    display(qdf)
    print(f"\nLossless: {'VERIFIED' if all(r['match_rate']==1.0 for r in quality_results_v2) else 'CHECK ABOVE'}")
 